In [1]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    print("文件夹:", dirname)
    for filename in filenames:
        print("  文件:", filename)

文件夹: /kaggle/input
文件夹: /kaggle/input/competitions
文件夹: /kaggle/input/competitions/house-prices-advanced-regression-techniques
  文件: sample_submission.csv
  文件: data_description.txt
  文件: train.csv
  文件: test.csv


In [2]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    print("文件夹:", dirname)
    for filename in filenames:
        print("  文件:", filename)


文件夹: /kaggle/input
文件夹: /kaggle/input/competitions
文件夹: /kaggle/input/competitions/house-prices-advanced-regression-techniques
  文件: sample_submission.csv
  文件: data_description.txt
  文件: train.csv
  文件: test.csv


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

# 1. 数据路径
data_path = Path("/kaggle/input/competitions/house-prices-advanced-regression-techniques")

# 2. 读取训练集和测试集
train = pd.read_csv(data_path / "train.csv")
test = pd.read_csv(data_path / "test.csv")

print("训练集大小:", train.shape)
print("测试集大小:", test.shape)

# 3. 拆分特征和标签
# SalePrice 是要预测的房价
y = np.log1p(train["SalePrice"])

X = train.drop(["SalePrice", "Id"], axis=1)
X_test = test.drop(["Id"], axis=1)

# 4. 区分数值特征和类别特征
num_cols = X.select_dtypes(exclude="object").columns
cat_cols = X.select_dtypes(include="object").columns

# 5. 数值特征处理：缺失值填中位数，然后标准化
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

# 6. 类别特征处理：缺失值填最常见值，然后独热编码
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 7. 合并预处理
preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# 8. 模型
model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", Ridge(alpha=10))
])

# 9. 交叉验证，看看大概效果
scores = -cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

print("每折 RMSE:", scores)
print("平均 RMSE:", scores.mean())

# 10. 用全部训练数据训练模型
model.fit(X, y)

# 11. 预测测试集
pred_log = model.predict(X_test)

# 12. 把 log 房价还原成真实房价
pred = np.expm1(pred_log)

# 防止极端情况下出现负数
pred = np.maximum(pred, 0)

# 13. 生成提交文件
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": pred
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("submission.csv 已生成")
submission.head()

训练集大小: (1460, 81)
测试集大小: (1459, 80)
每折 RMSE: [0.11695314 0.14848606 0.13106724 0.11807683 0.18553509]
平均 RMSE: 0.14002367247855108
submission.csv 已生成


,Id,SalePrice
0,1461,114299.055554
1,1462,145497.253328
2,1463,170686.166677
3,1464,192638.867823
4,1465,198263.893352
